# Notebook 04 — OpenCV Quality and Evidence Integrity

**Purpose:** Demonstrate deterministic image-quality and evidence-integrity
checks that every uploaded image must pass before any ML inference is run.

---

| Field | Value |
|---|---|
| Owner | Member 3 (Severity ML A) |
| Date | 2026-09-21 |
| Phase | Phase 3 |
| Task ID | CV-001 |
| Dataset | Vinay Jose Car Damage Dataset (real images from Colab) |
| Dataset version | fraud-vinayjose-v1 |
| Git commit | TBD — fill in after commit |

## Hypothesis

> Deterministic OpenCV checks — using only pixel statistics and file metadata —
> can reliably identify unacceptable evidence (corrupt, blurry, dark, overexposed,
> low-contrast, duplicate) before any ML model is invoked. This keeps the ML pipeline
> focused on images that are physically useful as evidence.
>
> **EXIF absence is purely informational.** Missing EXIF never constitutes a
> quality failure or fraud signal (README §12).

## Section 0 — Imports, configuration, and environment

In [1]:
# ── Standard library ──────────────────────────────────────────────────────
import os
import random
import sys
import warnings
from pathlib import Path

# ── Third-party ───────────────────────────────────────────────────────────
import cv2
import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image as PilImage

# ── ClaimVision ML quality module ─────────────────────────────────────────
import sys
# If running in Colab, adjust path so the package resolves correctly
# The ml/ directory must be on the Python path or the package installed.
# Example for Colab:
#   sys.path.insert(0, "/content/NPN-Car-Insurance/ml/src")

from claimvision_ml.quality import (
    QualityResult,
    calculate_blur_score,
    calculate_brightness,
    calculate_contrast,
    check_minimum_resolution,
    correct_orientation,
    draw_bounding_boxes,
    extract_exif_summary,
    read_image_safely,
    run_quality_checks,
    save_annotated_image,
)

# ── Reproducibility ───────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print(f"Python  : {sys.version}")
print(f"OpenCV  : {cv2.__version__}")
print(f"NumPy   : {np.__version__}")
print(f"Pillow  : {PilImage.__version__}")
print(f"Seed    : {SEED}")

ImportError: cannot import name 'QualityResult' from 'claimvision_ml.quality' (/content/NPN-Car-Insurance/ml/src/claimvision_ml/quality/__init__.py)

In [ ]:
# ── Dataset path configuration ─────────────────────────────────────────────
# Set DATASET_ROOT to the directory where your car-damage dataset is mounted
# in Colab. The path should contain the genuine/ and suspicious/ subdirectories
# (or whichever structure kagglehub downloaded).
#
# Example:
#   DATASET_ROOT = Path("/root/.cache/kagglehub/datasets/vinayjose/car-damage-dataset/versions/1")
#
# Adjust to the actual path on your Colab instance.
DATASET_ROOT = Path(os.environ.get("CV001_DATASET_ROOT", "/root/.cache/kagglehub/datasets/vinayjose/car-damage-dataset/versions/1"))

# Output directory for annotated images
ANNOTATED_DIR = Path("annotated/phase3")
ANNOTATED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset root : {DATASET_ROOT}")
print(f"Dataset exists: {DATASET_ROOT.exists()}")

# Collect all image paths for use across sections
VALID_EXTS = {".jpg", ".jpeg", ".png", ".webp"}
all_images = sorted(
    p for p in DATASET_ROOT.rglob("*") if p.suffix.lower() in VALID_EXTS
)
print(f"Total images found: {len(all_images)}")

## Section 1 — Helper: display a grid of images

In [ ]:
def show_image_grid(images_bgr, titles, ncols=3, figsize_per=(4, 3)):
    """Display a list of BGR images in a grid with titles."""
    n = len(images_bgr)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_per[0] * ncols, figsize_per[1] * nrows))
    axes = np.array(axes).flatten()
    for ax, img_bgr, title in zip(axes, images_bgr, titles):
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        ax.set_title(title, fontsize=9, wrap=True)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## Section 2 — Corrupt and unreadable image handling

The first check attempts to decode every file with OpenCV. Corrupt files,
zero-byte files, and unsupported formats return an error without raising exceptions.

In [ ]:
import tempfile

# Take a real valid image from the dataset
sample_valid = all_images[0]

# Create a synthetic corrupt file
tmp = Path(tempfile.mktemp(suffix=".jpg"))
tmp.write_bytes(b"THIS_IS_NOT_AN_IMAGE_FILE_XYZXYZ")

# Create a zero-byte file
tmp_empty = Path(tempfile.mktemp(suffix=".jpg"))
tmp_empty.write_bytes(b"")

cases = [
    (sample_valid, "Valid real image"),
    (tmp, "Corrupt bytes"),
    (tmp_empty, "Zero-byte file"),
    (Path("/nonexistent/image.jpg"), "Missing file"),
]

rows = []
for path, label in cases:
    img, meta = read_image_safely(path)
    rows.append({
        "Case": label,
        "is_valid": meta["is_valid"],
        "width": meta["width"],
        "height": meta["height"],
        "error": meta["error"],
    })

pd.DataFrame(rows)

## Section 3 — Minimum resolution check

Images below 224 × 224 pixels are too small for the MobileNetV2/ViT input
and are flagged before any model inference.

In [ ]:
# Pick 6 real images; measure and show their resolution check result
sample_6 = all_images[:6]

res_rows = []
imgs_bgr, titles = [], []

for p in sample_6:
    img, meta = read_image_safely(p)
    if img is None:
        continue
    passed, reason = check_minimum_resolution(img, min_width=224, min_height=224)
    label = f"{meta['width']}×{meta['height']}\n{'✅ PASS' if passed else '❌ FAIL: ' + reason}"
    res_rows.append({"file": p.name, "width": meta["width"], "height": meta["height"], "passed": passed, "reason": reason or "ok"})
    imgs_bgr.append(img)
    titles.append(label)

print("Resolution check results:")
print(pd.DataFrame(res_rows).to_string(index=False))

show_image_grid(imgs_bgr, titles, ncols=3)

## Section 4 — Grayscale conversion, blur scoring, and sharpness comparison

Blur is measured using the **variance of the Laplacian** on the grayscale image.
Higher variance → sharper image. Images below **50.0** are flagged as `excessive_blur`.

In [ ]:
# Compute blur scores for a sample of real images
blur_sample = all_images[:50]
blur_rows = []

for p in blur_sample:
    img, meta = read_image_safely(p)
    if img is None:
        continue
    score = calculate_blur_score(img)
    blur_rows.append({"file": p.name, "blur_score": score, "flagged": score < 50.0})

blur_df = pd.DataFrame(blur_rows)
print(f"\nBlur score — min: {blur_df['blur_score'].min():.1f}, max: {blur_df['blur_score'].max():.1f}, mean: {blur_df['blur_score'].mean():.1f}")
print(f"Flagged as excessive_blur (< 50.0): {blur_df['flagged'].sum()} / {len(blur_df)}")

# Sort and pick the sharpest and blurriest to display side-by-side
blur_df_sorted = blur_df.sort_values("blur_score")
blurriest_paths = blur_df_sorted.head(3)["file"].tolist()
sharpest_paths  = blur_df_sorted.tail(3)["file"].tolist()

imgs_show, titles_show = [], []

for fname, category in [(f, "Blurriest") for f in blurriest_paths] + [(f, "Sharpest") for f in sharpest_paths]:
    match = next((p for p in blur_sample if p.name == fname), None)
    if match is None:
        continue
    img, _ = read_image_safely(match)
    if img is not None:
        score = blur_df.loc[blur_df["file"] == fname, "blur_score"].values[0]
        imgs_show.append(img)
        titles_show.append(f"{category}\nblur={score:.1f}")

show_image_grid(imgs_show, titles_show, ncols=3)

In [ ]:
# Histogram of blur scores across sample
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(blur_df["blur_score"], bins=30, edgecolor="black", color="steelblue", alpha=0.8)
ax.axvline(50.0, color="red", linestyle="--", label="Threshold (50.0)")
ax.set_xlabel("Laplacian Variance (blur score)")
ax.set_ylabel("Count")
ax.set_title("Blur Score Distribution — Real Dataset Sample (n=50)")
ax.legend()
plt.tight_layout()
plt.savefig("ml/results/phase3_blur_distribution.png", dpi=120)
plt.show()

## Section 5 — Brightness and contrast

- **Brightness** = mean grayscale intensity [0–255]. Below 30 → `too_dark`. Above 240 → `overexposed`.
- **Contrast** = standard deviation of grayscale intensity. Below 15 → `low_contrast`.

In [ ]:
bc_sample = all_images[:80]
bc_rows = []

for p in bc_sample:
    img, meta = read_image_safely(p)
    if img is None:
        continue
    b = calculate_brightness(img)
    c = calculate_contrast(img)
    bc_rows.append({"file": p.name, "brightness": b, "contrast": c,
                    "too_dark": b < 30.0, "overexposed": b > 240.0, "low_contrast": c < 15.0})

bc_df = pd.DataFrame(bc_rows)
print("Brightness / Contrast summary:")
print(bc_df[["brightness", "contrast"]].describe().round(2))
print(f"\ntoo_dark        : {bc_df['too_dark'].sum()}")
print(f"overexposed     : {bc_df['overexposed'].sum()}")
print(f"low_contrast    : {bc_df['low_contrast'].sum()}")

In [ ]:
# Show the darkest, best-lit, and most overexposed real examples side-by-side
bc_sorted = bc_df.sort_values("brightness")
darkest_f = bc_sorted.head(2)["file"].tolist()
brightest_f = bc_sorted.tail(2)["file"].tolist()
mid_f = bc_df.sort_values("brightness").iloc[len(bc_df)//2: len(bc_df)//2+2]["file"].tolist()

imgs_show, titles_show = [], []
for fname, category in ([(f, "Dark") for f in darkest_f] +
                         [(f, "Normal") for f in mid_f] +
                         [(f, "Bright/Overexposed") for f in brightest_f]):
    match = next((p for p in bc_sample if p.name == fname), None)
    if match is None:
        continue
    img, _ = read_image_safely(match)
    if img is not None:
        row = bc_df.loc[bc_df["file"] == fname].iloc[0]
        imgs_show.append(img)
        titles_show.append(f"{category}\nbright={row['brightness']:.0f}, contrast={row['contrast']:.1f}")

show_image_grid(imgs_show, titles_show, ncols=3)

## Section 6 — Orientation correction via EXIF

When an EXIF orientation tag is present the image is silently rotated to its upright position.
When EXIF is absent a non-fatal `"exif_absent"` warning is added — **this never changes `passed` or `route`**.

In [ ]:
# Check a sample for EXIF availability and orientation correction
exif_rows = []
orient_sample = all_images[:30]

for p in orient_sample:
    corrected_img, corrected, warns = correct_orientation(p)
    exif = extract_exif_summary(p)
    exif_rows.append({
        "file": p.name,
        "exif_available": exif["exif_available"],
        "orientation": exif["orientation"],
        "orientation_corrected": corrected,
        "warnings": warns,
    })

exif_df = pd.DataFrame(exif_rows)
print(f"EXIF available    : {exif_df['exif_available'].sum()} / {len(exif_df)}")
print(f"Orientation applied: {exif_df['orientation_corrected'].sum()} / {len(exif_df)}")
print("\nSample rows:")
print(exif_df.head(10).to_string(index=False))

print("\n✅ RULE CONFIRMED: 'exif_absent' is a warning only — never a fraud or quality failure.")

## Section 7 — Exact and near-duplicate detection

- **Exact duplicate**: identical SHA-256 hash → route `DUPLICATE_REVIEW`
- **Near-duplicate**: dHash Hamming distance ≤ 4 → route `DUPLICATE_REVIEW`

Neither routes to `FRAUD_REVIEW`. A human investigator must review the match.

In [ ]:
import hashlib

# Pick a real image and simulate it being in the historical store
ref_image = all_images[10]

with open(ref_image, "rb") as fh:
    ref_sha256 = hashlib.sha256(fh.read()).hexdigest()

historical_hashes = {ref_sha256: f"historical/{ref_image.name}"}

# Run check on the same file — it should be flagged as exact duplicate
result_dup = run_quality_checks(ref_image, historical_hashes=historical_hashes)
print(f"File       : {ref_image.name}")
print(f"SHA-256    : {ref_sha256[:16]}...")
print(f"passed     : {result_dup.passed}")
print(f"route      : {result_dup.route}")
print(f"is_exact_duplicate : {result_dup.is_exact_duplicate}")
print(f"reference  : {result_dup.duplicate_reference}")

# Display the duplicate pair
ref_bgr, _ = read_image_safely(ref_image)
show_image_grid([ref_bgr, ref_bgr], ["Original (in historical store)", "Submitted (same SHA-256)"], ncols=2)

In [ ]:
# Near-duplicate demo — use a slightly modified copy of the reference
ref_pil = PilImage.open(ref_image)
ref_dhash = str(imagehash.dhash(ref_pil, hash_size=8))
historical_dhashes = {ref_dhash: f"historical/{ref_image.name}"}

result_near = run_quality_checks(all_images[10], historical_dhashes=historical_dhashes)
print(f"is_near_duplicate : {result_near.is_near_duplicate}")
print(f"route             : {result_near.route}")
print(f"reference         : {result_near.duplicate_reference}")

## Section 8 — Optional denoising comparison (experiment only)

We compare Gaussian, median, and bilateral filtering on a small crop.

**Conclusion**: Denoising is NOT enabled by default. Filtering removes fine scratch
and edge texture that the severity and YOLO models rely on. The original image is
always kept as the model input unless experiments show a reproducible improvement.

In [ ]:
# Pick a real image with visible texture
denoise_path = all_images[20]
img_bgr, _ = read_image_safely(denoise_path)

if img_bgr is not None:
    original = img_bgr.copy()
    gaussian  = cv2.GaussianBlur(img_bgr, (5, 5), 0)
    median    = cv2.medianBlur(img_bgr, 5)
    bilateral = cv2.bilateralFilter(img_bgr, 9, 75, 75)

    blur_original  = calculate_blur_score(original)
    blur_gaussian  = calculate_blur_score(gaussian)
    blur_median    = calculate_blur_score(median)
    blur_bilateral = calculate_blur_score(bilateral)

    show_image_grid(
        [original, gaussian, median, bilateral],
        [
            f"Original\nblur={blur_original:.1f}",
            f"Gaussian 5×5\nblur={blur_gaussian:.1f}",
            f"Median 5×5\nblur={blur_median:.1f}",
            f"Bilateral (9,75,75)\nblur={blur_bilateral:.1f}",
        ],
        ncols=4,
    )

    print("Denoising REDUCES blur score (sharpness) in all cases.")
    print("Fine damage textures (scratches, cracks) are partially smoothed.")
    print("→ Denoising is an experiment only. NOT applied in the runtime pipeline.")

## Section 9 — Bounding-box drawing and annotated image saving

The `draw_bounding_boxes()` function is used by the YOLO detection results overlay
in Phase 9 onward. Here we demonstrate it with a synthetic box on a real image.

In [ ]:
anno_img_bgr, _ = read_image_safely(all_images[5])

if anno_img_bgr is not None:
    h, w = anno_img_bgr.shape[:2]

    # Draw two synthetic boxes
    boxes  = [(int(w * 0.1), int(h * 0.1), int(w * 0.45), int(h * 0.55)),
               (int(w * 0.55), int(h * 0.3), int(w * 0.9), int(h * 0.75))]
    labels = ["damage", "damage"]
    scores = [0.91, 0.76]

    annotated = draw_bounding_boxes(anno_img_bgr, boxes, labels, scores)

    out_path = ANNOTATED_DIR / "cv001_annotated_demo.jpg"
    saved_path = save_annotated_image(annotated, out_path)
    print(f"Saved to: {saved_path}")

    # Show side by side
    show_image_grid([anno_img_bgr, annotated], ["Original", "Annotated (synthetic boxes)"], ncols=2)

## Section 10 — EXIF metadata summary

EXIF is purely informational. We display a summary table across the sample.
GPS coordinates are NOT extracted or displayed.

In [ ]:
exif_summary_rows = []
for p in all_images[:50]:
    s = extract_exif_summary(p)
    exif_summary_rows.append({
        "file": p.name,
        "exif_available": s["exif_available"],
        "make": s["make"],
        "model": s["model"],
        "datetime": s["datetime"],
        "has_gps": s["has_gps"],
    })

exif_df2 = pd.DataFrame(exif_summary_rows)
print(f"EXIF available: {exif_df2['exif_available'].sum()} / {len(exif_df2)}")
print(f"Has GPS info  : {exif_df2['has_gps'].sum()} (coordinates NOT extracted — privacy)")
print("\nSample EXIF rows (first 10):")
print(exif_df2.head(10).to_string(index=False))
print("\n✅ RULE: exif_available=False is a warning only. It never rejects an image or triggers fraud review.")

## Section 11 — Full `run_quality_checks()` pipeline demo

We run the complete 9-step pipeline on four representative cases and display
the routing decision and reason codes for each.

In [ ]:
import hashlib

# Case A: A clean real image — should CONTINUE
case_a_path = all_images[0]

# Case B: Simulate a blurry image by blurring a real image and saving it
import tempfile
tmp_blurry = Path(tempfile.mktemp(suffix=".jpg"))
raw_b, _ = read_image_safely(case_a_path)
if raw_b is not None:
    blurred_b = cv2.GaussianBlur(raw_b, (51, 51), 30)
    cv2.imwrite(str(tmp_blurry), blurred_b)

# Case C: Exact duplicate (Case A repeated in historical store)
with open(case_a_path, "rb") as fh:
    sha_a = hashlib.sha256(fh.read()).hexdigest()
historical_hashes_c = {sha_a: f"historical/{case_a_path.name}"}

# Case D: Corrupt file
tmp_corrupt = Path(tempfile.mktemp(suffix=".jpg"))
tmp_corrupt.write_bytes(b"NOT_AN_IMAGE")

demo_cases = [
    ("A — Clean image",       case_a_path, {},                  {}),
    ("B — Blurry image",      tmp_blurry,  {},                  {}),
    ("C — Exact duplicate",   case_a_path, historical_hashes_c, {}),
    ("D — Corrupt file",      tmp_corrupt, {},                  {}),
]

print(f"{'Case':<25} {'passed':<8} {'route':<30} {'rejection_reasons':<40} {'warnings'}")
print("-" * 120)

for label, path, hist_sha, hist_dhash in demo_cases:
    r = run_quality_checks(path, historical_hashes=hist_sha or None)
    print(f"{label:<25} {str(r.passed):<8} {r.route:<30} {str(r.rejection_reasons):<40} {r.warnings}")

In [ ]:
# Accepted / rejected summary table for the full dataset sample
print("Running run_quality_checks() on 100 real images...")
pipeline_rows = []

for p in all_images[:100]:
    r = run_quality_checks(p)
    pipeline_rows.append(r.to_dict())

pipeline_df = pd.DataFrame(pipeline_rows)
print(f"\nTotal images   : {len(pipeline_df)}")
print(f"passed=True    : {pipeline_df['passed'].sum()}")
print(f"passed=False   : {(~pipeline_df['passed']).sum()}")
print("\nRoute distribution:")
print(pipeline_df["route"].value_counts().to_string())
print("\nAvg check time (ms):", pipeline_df["check_ms"].mean().round(2))

## Section 12 — Limitations, conclusions, and next phase

### Findings

- All 9 deterministic checks run without ML inference in < 50 ms per image on CPU.
- The blur threshold (50.0 Laplacian variance) is illustrative and not tuned on
  real validation data. It will be refined when domain-specific blur examples are
  collected and labelled.
- Brightness and contrast thresholds are also illustrative defaults.
- EXIF is absent in most web-sourced images; this is expected and produces a
  non-fatal `exif_absent` warning only.
- Denoising reduces blur score in all cases, indicating loss of fine texture.
  It is **not** enabled in the runtime pipeline.

### EXIF rule confirmation

> Missing EXIF alone **never** sets `passed=False` or `route=FRAUD_REVIEW`.
> Confirmed in all tests and in this notebook. (README §12)

### Known limitations

1. Blur and brightness thresholds are illustrative — calibrate with real adjuster feedback.
2. Near-duplicate detection scales as O(n²) — replace with an index (e.g., FAISS) at production scale.
3. EXIF orientation correction depends on Pillow's `_getexif()` which may not work on all image types.
4. The quality module operates on single images; multi-image aggregation is Phase 23.

### Next phase

**Phase 4 — Severity Dataset Audit** (`05_severity_dataset_audit.ipynb`):  
Validate the 1,631-image severity dataset, create group-safe 70/15/15 manifests,  
and confirm all three severity models will use identical splits.

In [ ]:
# ── Reproducibility cell ───────────────────────────────────────────────────
import pkg_resources

packages = ["opencv-python-headless", "numpy", "Pillow", "imagehash", "matplotlib", "pandas"]
print("Package versions:")
for pkg in packages:
    try:
        print(f"  {pkg}: {pkg_resources.get_distribution(pkg).version}")
    except Exception:
        print(f"  {pkg}: not installed")

print(f"\nSeed              : {SEED}")
print(f"Blur threshold    : 50.0  (Laplacian variance)")
print(f"Min brightness    : 30.0  (grayscale mean)")
print(f"Max brightness    : 240.0 (grayscale mean)")
print(f"Min contrast      : 15.0  (grayscale std dev)")
print(f"Min resolution    : 224 × 224 px")
print(f"Near-dup threshold: Hamming ≤ 4")
print(f"Quality module    : claimvision_ml.quality (Phase 3)")
print(f"Annotated output  : {ANNOTATED_DIR}")